# 📗 OpenAI API 활용 — 이미지 이해와 정형화

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간에 **문장**을 표로 만들었습니다. 이번 시간엔 **사진**으로 같은 일을 합니다. 요즘 모델은 글과 그림을 함께 읽습니다(**비전**). 사진을 넣고 "무엇이 있나"를 묻는 것에서 시작해, **패션 사진 → 상품 속성 표**, **영수증 사진 → 품목·금액 표** 까지 만들어 봅니다.

핵심은 지난 시간과 **똑같습니다** — 달라지는 건 `content` 에 텍스트뿐 아니라 **이미지가 함께 들어간다**는 것뿐입니다.

## ⏪ 복습 — 지난 시간

- `parse(response_format=스키마)` 로 자유 텍스트를 **표**로 만들었습니다.
- `Optional` 로 **모르는 값은 비우고**, `list[...]` 로 **여러 건을 중첩**해 받았습니다.
- 오늘은 입력이 텍스트에서 **이미지**로 바뀔 뿐, 스키마 설계는 그대로 쓰입니다.

**오늘의 목표**

- [ ] 이미지를 모델에 넣는 **두 가지 방법**(로컬 파일 base64 / 공개 URL)을 구분해 쓴다.
- [ ] **여러 장을 한 요청**에 넣어 비교시킨다.
- [ ] 패션 사진을 **스키마로 정형화**해 상품 속성 표를 만든다.
- [ ] 영수증 사진에서 **품목·금액을 읽어(OCR)** 표로 만들고, 합계로 **검증**한다.
- [ ] (참고) MCP·웹검색이 무엇인지 감을 잡는다.

아래 준비 셀들을 먼저 실행하세요(키가 없어도 저장된 응답으로 진행됩니다).

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요.
# .env 에 OPENAI_API_KEY 가 있으면 실제 OpenAI 에 연결하고,
# 없으면 미리 저장해 둔 응답(data/api_cache.json)으로 진행됩니다(키·인터넷 없이 실습 가능).
# 아래 client 사용법은 공식 문서와 똑같습니다 → https://developers.openai.com/api/docs/guides/text
import sys
sys.path.insert(0, '.')          # openai_client.py 가 있는 폴더

from openai_client import get_client

client = get_client()

In [ ]:
# [제공 코드] 라이브러리
import base64
import json
import pandas as pd
from pydantic import BaseModel, Field
from typing import Literal, Optional

---
# 1. 이미지를 넣는 두 가지 방법

## 왜 두 가지인가
모델은 우리 컴퓨터 안을 볼 수 없습니다. 그래서 이미지를 **요청에 실어 보내거나**, **인터넷에서 가져올 주소를 알려 주거나** 둘 중 하나를 해야 합니다.

| | 방법① 로컬 파일 | 방법② 공개 URL |
|---|---|---|
| 어떻게 | 파일을 읽어 **base64 문자열**로 바꿔 함께 보낸다 | 이미지 **주소만** 넘긴다 |
| 쓸 때 | 내 컴퓨터·사내 서버의 사진(대부분의 실습·업무) | 이미 웹에 공개된 사진 |
| 장점 | 인터넷에 공개하지 않아도 된다 | 요청이 가볍다(주소 한 줄) |
| 단점 | 요청이 커진다(파일 크기만큼) | **공개 접근 가능해야** 한다. 주소가 죽으면 실패 |


<img src="images/이미지입력_파일_vs_URL.jpg" width="820">

*왼쪽(방법①): 사진을 **통째로 실어** 보낸다 — 요청이 무거워지는 대신 공개할 필요가 없다. 오른쪽(방법②): **주소만** 건네면 서버가 직접 가져온다 — 가벼운 대신 그 주소에 접근할 수 있어야 한다.*

두 방법 모두 `content` 를 **리스트**로 만들고, 그 안에 `{'type': 'text', ...}` 와 `{'type': 'image_url', ...}` 를 나란히 넣습니다. 지금까지 `content` 에 문자열만 넣던 것과 다른 점은 이것뿐입니다.

## 방법① 로컬 파일 — base64 로 실어 보내기

In [ ]:
def to_data_url(path):
    """로컬 이미지 파일을 모델에 넣을 수 있는 data URL 문자열로 바꾼다."""
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'data:image/jpeg;base64,{b64}'

food_url = to_data_url('data/images/food.jpg')
print('data URL 앞부분:', food_url[:50], '...')
print('전체 길이:', len(food_url), '자  ← 파일이 통째로 문자열이 되어 함께 전송된다')

In [ ]:
resp = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': [
        {'type': 'text', 'text': '이 사진에 무엇이 있는지 한 문장으로 설명해줘.'},
        {'type': 'image_url', 'image_url': {'url': food_url}}]}],
    max_tokens=200)
print(resp.choices[0].message.content)

## 방법② 공개 URL — 주소만 넘기기

In [ ]:
# 위키미디어 공용에 공개된 사진(커피 한 잔)
PUBLIC_IMAGE = 'https://upload.wikimedia.org/wikipedia/commons/4/45/A_small_cup_of_coffee.JPG'

resp = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': [
        {'type': 'text', 'text': '이 사진에 무엇이 있는지 한 문장으로 설명해줘.'},
        {'type': 'image_url', 'image_url': {'url': PUBLIC_IMAGE}}]}],
    max_tokens=200)
print(resp.choices[0].message.content)

> 코드 모양이 **똑같습니다** — `image_url` 의 `url` 자리에 **data URL** 을 넣느냐 **웹 주소**를 넣느냐만 다릅니다. 실무에서는 대부분 방법①(로컬 파일)을 씁니다. 사내 이미지를 공개할 수는 없으니까요.

> ⚠️ 방법②는 **모델 서버가 그 주소로 직접 접속**합니다. 로그인이 필요한 주소, 사내망 주소, 핫링크를 막아 둔 주소는 실패합니다.

### 🖐️ 함께 따라하기 — 다른 사진을 파일로 넣어 보기

`data/images/` 폴더의 다른 사진(`ad.jpg`)을 방법①로 넣어 무엇이 보이는지 물어봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) to_data_url('data/images/ad.jpg') 로 data URL 을 만든다
# 2) content 를 [텍스트, image_url] 리스트로 만들어 gpt-4o-mini 를 호출한다
#    질문: '이 이미지는 무엇을 광고하고 있나요? 한 문장으로.'
# 3) 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 사내 서버에만 있는 사진을 분석하려면 두 방법 중 무엇을 써야 하나요?

<details><summary>정답 보기</summary>

**방법①(로컬 파일 → base64)** 입니다. 방법②는 모델 서버가 그 주소로 직접 접속할 수 있어야 하므로 사내망 주소로는 동작하지 않습니다.

</details>

**2.** 이미지를 넣을 때 `content` 의 자료형이 지금까지와 어떻게 달라지나요?

<details><summary>정답 보기</summary>

문자열 하나가 아니라 **리스트**가 됩니다. 그 안에 `{'type':'text',…}` 와 `{'type':'image_url',…}` 를 나란히 넣습니다.

</details>

---
# 2. 여러 장을 한 번에 — 비교시키기

## 왜 필요할까요?
"이 두 광고 중 어느 쪽이 더 눈에 띄나", "이번 촬영본과 지난 촬영본의 차이는" 같은 질문은 **한 장씩 봐서는 답할 수 없습니다.** 이미지를 **같은 요청에 여러 개** 넣으면 모델이 나란히 놓고 비교합니다.

방법은 간단합니다 — `content` 리스트에 `image_url` 항목을 **여러 개** 넣으면 됩니다.

In [ ]:
urls = [to_data_url('data/images/ad.jpg'), to_data_url('data/images/scene.jpg')]

content = [{'type': 'text', 'text': '두 이미지의 분위기와 쓰임새가 어떻게 다른지 각각 한 문장으로 비교해줘.'}]
for u in urls:
    content.append({'type': 'image_url', 'image_url': {'url': u}})

resp = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': content}], max_tokens=300)
print(resp.choices[0].message.content)

> 이미지를 넣을수록 **요청이 커지고 요금이 올라갑니다.** 이미지는 크기에 비례해 토큰으로 환산되므로, 필요 없는 큰 사진은 미리 줄여서 보내는 것이 좋습니다(오늘 쓰는 사진들도 긴 변 1400px 로 줄여 두었습니다).

### ✅ 바로 확인 퀴즈

**1.** 사진 세 장을 한 번에 비교시키려면 무엇을 바꾸면 되나요?

<details><summary>정답 보기</summary>

`content` 리스트에 **`image_url` 항목을 세 개** 넣으면 됩니다. 나머지 코드는 그대로입니다.

</details>

---
# 3. 패션 사진을 상품 속성 표로

## 왜 필요할까요?
쇼핑몰에는 상품 사진 수만 장이 있는데 **속성(색·카테고리·스타일)은 비어 있는** 경우가 많습니다. 사람이 하나씩 입력하려면 끝이 없습니다. 사진을 넣고 **스키마로 받으면** 곧장 상품 DB 에 넣을 표가 됩니다.

지난 시간의 정형화와 **완전히 같은 구조**입니다 — 입력이 문장에서 사진으로 바뀌었을 뿐입니다.

In [ ]:
class FashionItem(BaseModel):
    """사진 속 착장 아이템 하나."""
    item_type: Literal['상의', '하의', '아우터', '원피스', '신발', '가방', '액세서리'] = Field(
        description='아이템 종류')
    color: Literal['블랙', '화이트', '그레이', '네이비', '블루', '레드', '베이지', '브라운',
                    '그린', '옐로우', '핑크', '기타'] = Field(description='가장 두드러진 색 하나')
    pattern: Literal['무지', '스트라이프', '체크', '프린트', '기타'] = Field(description='무늬')

class FashionPhoto(BaseModel):
    """패션 사진 한 장의 분석 결과."""
    gender: Literal['남성', '여성', '공용'] = Field(description='착장의 대상 성별')
    style: Literal['캐주얼', '포멀', '스포티', '스트리트', '기타'] = Field(description='전체 스타일')
    items: list[FashionItem] = Field(description='사진에서 보이는 아이템들. 최대 5개')

def analyze_fashion(path):
    resp = client.chat.completions.parse(model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': [
            {'type': 'text', 'text': '이 패션 사진을 스키마에 맞춰 분석해줘.'},
            {'type': 'image_url', 'image_url': {'url': to_data_url(path)}}]}],
        response_format=FashionPhoto, max_tokens=800)
    return resp.choices[0].message.parsed

In [ ]:
# 폴더의 사진 12장 중 앞 4장으로 시연한다(호출 비용을 아끼려면 먼저 소수로 확인하는 것이 실무 습관)
from pathlib import Path
fashion_paths = sorted(Path('data/fashion').glob('*.jpg'))[:4]
print('분석할 사진:', [p.name[:28] + '…' for p in fashion_paths])

rows = []
for p in fashion_paths:
    r = analyze_fashion(p)
    for it in r.items:
        rows.append({'image_file': p.name, 'gender': r.gender, 'style': r.style,
                     'item_type': it.item_type, 'color': it.color, 'pattern': it.pattern})

fashion_df = pd.DataFrame(rows)
display(fashion_df)
print('사진 4장에서 아이템', len(fashion_df), '개 추출')

사진 4장이 **아이템 단위 행**으로 펼쳐졌습니다(사진 하나에 아이템 여러 개 → `list[FashionItem]` 덕분). 이제 색·종류별 집계가 바로 됩니다.

In [ ]:
print('[아이템 종류별]')
display(fashion_df['item_type'].value_counts().to_frame('개수'))
print('[색상별]')
display(fashion_df['color'].value_counts().to_frame('개수'))

> **사람이 붙인 정답표와 대조해 보기** — 이 데이터에는 사람이 직접 라벨링한 `data/fashion_items.csv` 가 함께 있습니다. 모델이 뽑은 아이템 수와 사람이 적은 아이템 수를 비교하면 **빠뜨린 것이 있는지**를 알 수 있습니다.

In [ ]:
truth = pd.read_csv('data/fashion_items.csv')
names = [p.name for p in fashion_paths]
truth_n = truth[truth['image_file'].isin(names)].groupby('image_file').size()
model_n = fashion_df.groupby('image_file').size()
compare = pd.DataFrame({'사람이 적은 아이템 수': truth_n, '모델이 찾은 아이템 수': model_n}).fillna(0).astype(int)
display(compare)
print('※ 수가 다르다고 틀린 것은 아닙니다 — 사람은 액세서리를 생략하기도, 모델은 그림자를 아이템으로 보기도 합니다.')

### 🖐️ 함께 따라하기 — 다섯 번째 사진 분석

같은 함수로 다음 사진 한 장을 더 분석해, 스타일과 아이템이 어떻게 나오는지 확인합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) sorted(Path('data/fashion').glob('*.jpg'))[4] 로 다섯 번째 사진 경로를 얻는다
# 2) analyze_fashion() 으로 분석해 r 에 담는다
# 3) r.gender, r.style 과 각 아이템의 item_type·color 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 사진 한 장에서 아이템이 여러 개 나오는 것을 스키마로 어떻게 표현했나요?

<details><summary>정답 보기</summary>

**`items: list[FashionItem]`** — 지난 시간의 개인정보 `items` 와 같은 **중첩 리스트** 구조입니다.

</details>

**2.** 12장 전부가 아니라 4장으로 먼저 돌려 본 이유는?

<details><summary>정답 보기</summary>

**스키마와 프롬프트가 의도대로 동작하는지 싸게 확인**하기 위해서입니다. 스키마가 잘못된 채 수천 장을 돌리면 시간과 요금을 그대로 버리게 됩니다.

</details>

---
# 4. 영수증 사진에서 품목·금액 읽기 (OCR + 정형화)

## 왜 필요할까요?
경비 정산·가계부·매입 관리는 전부 **영수증 사진 → 표** 작업입니다. 예전에는 OCR 로 글자를 뽑고 그 글자를 다시 규칙으로 해석해야 했습니다. 지금은 **사진을 넣고 원하는 표 모양을 스키마로 주면** 한 번에 끝납니다.

영수증은 **한 장에 품목이 여러 줄**이므로 또 중첩 구조입니다.

```
Receipt
 ├─ store_name   : 가게 이름(안 보이면 null)
 ├─ total_amount : 총액
 ├─ payment      : 결제수단(현금·카드·기타)
 └─ items        : [ {name, quantity, price}, ... ]   ← 중첩
```

In [ ]:
class ReceiptItem(BaseModel):
    """영수증의 품목 한 줄."""
    name: str = Field(description='품목명. 영수증에 적힌 그대로')
    quantity: int = Field(description='수량. 안 적혀 있으면 1')
    price: float = Field(description='그 줄의 합계 금액(수량 × 단가)')

class Receipt(BaseModel):
    """영수증 사진 한 장에서 읽어 낸 결과."""
    store_name: Optional[str] = Field(default=None, description='가게 이름. 안 보이면 null')
    total_amount: float = Field(description='영수증에 적힌 총 결제 금액')
    payment: Literal['현금', '카드', '기타'] = Field(description='결제 수단')
    items: list[ReceiptItem] = Field(description='품목 목록')

def read_receipt(path):
    resp = client.chat.completions.parse(model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': [
            {'type': 'text', 'text': '이 영수증에서 가게·총액·결제수단·품목을 스키마대로 읽어줘. 안 보이는 값은 null.'},
            {'type': 'image_url', 'image_url': {'url': to_data_url(path)}}]}],
        response_format=Receipt, max_tokens=1500)
    return resp.choices[0].message.parsed

In [ ]:
receipt_paths = sorted(Path('data/receipt').glob('*.jpg'))[:4]

receipt_rows, item_rows = [], []
for p in receipt_paths:
    r = read_receipt(p)
    receipt_rows.append({'image_file': p.name, 'store_name': r.store_name,
                         'total_amount': r.total_amount, 'payment': r.payment,
                         'item_count': len(r.items)})
    for it in r.items:
        item_rows.append({'image_file': p.name, 'name': it.name,
                          'quantity': it.quantity, 'price': it.price})

receipt_df = pd.DataFrame(receipt_rows)
item_df = pd.DataFrame(item_rows)
display(receipt_df)
display(item_df)

## 읽은 값을 **검증**한다 — 영수증에는 정답이 숨어 있다

OCR 은 숫자를 잘못 읽습니다(0↔8, 1↔7). 그런데 영수증에는 **스스로를 검산할 수단**이 있습니다 — **품목 금액의 합 = 총액** 이어야 합니다. 이 검산은 코드로 자동화할 수 있고, **LLM 이 읽은 값을 그대로 믿지 않는 습관**의 출발점입니다.

In [ ]:
check = item_df.groupby('image_file')['price'].sum().rename('품목 합')
verify = receipt_df.set_index('image_file')[['total_amount']].join(check).fillna(0)
verify['차이'] = (verify['total_amount'] - verify['품목 합']).round(2)
verify['검산'] = verify['차이'].abs() < 1
display(verify)
print('검산 통과:', int(verify['검산'].sum()), '/', len(verify), '장')
print('※ 할인·부가세가 따로 적힌 영수증은 합이 안 맞을 수 있습니다 — 그런 자리를 찾아내는 것이 검산의 목적입니다.')

> 사람이 정리해 둔 정답표(`receipts.csv`)와도 대조해 봅니다. 총액이 맞는지 보면 이 파이프라인을 실제로 쓸 수 있는지 판단할 수 있습니다.

In [ ]:
truth_r = pd.read_csv('data/receipts.csv')[['image_file', 'store_name', 'total_amount']]
merged = receipt_df[['image_file', 'store_name', 'total_amount']].merge(
    truth_r, on='image_file', suffixes=('_모델', '_정답'))
merged['총액 일치'] = (merged['total_amount_모델'] - merged['total_amount_정답']).abs() < 1
display(merged)
print('총액 일치:', int(merged['총액 일치'].sum()), '/', len(merged), '장')

In [ ]:
# 결과를 저장한다 — 정산 시스템에 넣을 수 있는 형태
import os
os.makedirs('output', exist_ok=True)
receipt_df.to_csv('output/receipts_parsed.csv', index=False, encoding='utf-8-sig')
item_df.to_csv('output/receipt_items_parsed.csv', index=False, encoding='utf-8-sig')
print('저장 완료: output/receipts_parsed.csv, output/receipt_items_parsed.csv')

### 🖐️ 함께 따라하기 — 다섯 번째 영수증 읽기

영수증 한 장을 더 읽고, **품목 합과 총액이 맞는지** 직접 검산해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) sorted(Path('data/receipt').glob('*.jpg'))[4] 로 다섯 번째 영수증 경로를 얻는다
# 2) read_receipt() 로 읽어 r 에 담는다
# 3) 품목 금액의 합(sum)과 r.total_amount 를 각각 출력하고, 차이가 1원 미만인지 확인한다

### ✅ 바로 확인 퀴즈

**1.** 영수증에서 읽은 숫자를 검증하는 가장 간단한 방법은?

<details><summary>정답 보기</summary>

**품목 금액의 합과 총액을 비교**하는 것입니다. 두 값이 다르면 어딘가를 잘못 읽었거나 할인·세금이 따로 있는 영수증입니다.

</details>

**2.** 가게 이름이 사진에서 잘려 안 보일 때 스키마는 어떻게 동작해야 하나요?

<details><summary>정답 보기</summary>

`store_name` 이 **`Optional`** 이므로 **`None`** 이 와야 합니다. 그럴듯한 가게 이름을 지어내면 정산 데이터가 통째로 오염됩니다.

</details>

---
# 5. 참고 — MCP·웹검색 (맛보기)

아래는 **오늘 과제 범위는 아니지만** 앞으로 자주 만날 두 가지입니다. 개념만 가볍게 보고 넘어가세요.

## 5-1. MCP(Model Context Protocol) — 도구 연결의 표준 규격
2교시에서 Function Calling 으로 함수를 **매번 손으로** 정의했습니다. **MCP** 는 이런 도구·데이터 연결을 **표준 방식**으로 만든 규격입니다 — 여러 기기를 하나로 꽂는 **USB-C** 같은 것입니다. 한 번 만든 도구 서버(파일 접근·DB·검색)를 여러 앱이 **똑같이** 꽂아 쓸 수 있습니다.

- **Function Calling 과 관계**: MCP 는 Function Calling 을 **표준화·재사용**하기 쉽게 감싼 것입니다.
- **실제 MCP 서버 연동은 뒤 과목(에이전트)에서** 배웁니다.

## 5-2. 웹검색 — 최신 정보 가져오기
모델은 학습 시점 이후 소식을 모릅니다. **Responses API** 에 `tools=[{'type': 'web_search'}]` 를 주면 모델이 **웹을 검색해** 최신 정보를 반영해 답합니다.

In [ ]:
resp = client.responses.create(model='gpt-4o-mini',
    tools=[{'type': 'web_search'}],
    input='오늘 서울 날씨를 한 문장으로 알려줘.')
print(resp.output_text)

---
## 이번 강의 정리

| 주제 | 핵심 | 코드 |
|---|---|---|
| 이미지 입력 ① | 로컬 파일 → base64 data URL | `data:image/jpeg;base64,...` |
| 이미지 입력 ② | 공개 URL 을 그대로 | `image_url: {'url': 'https://...'}` |
| 여러 장 | `content` 리스트에 `image_url` 여러 개 | 비교·차이 분석 |
| 이미지 정형화 | 사진 → 스키마 → 표 | `parse(response_format=...)` |
| 검증 | 읽은 값을 스스로 검산 | 품목 합 == 총액 |

- 텍스트든 이미지든 **정형화의 문법은 같습니다** — 달라지는 건 `content` 에 무엇을 넣느냐뿐입니다.
- **모델이 읽은 값은 검증할 수 있으면 반드시 검증하세요.** 정산·회계처럼 틀리면 안 되는 데이터일수록 그렇습니다.

## ⏭️ 예고 — 다음 시간

다음 단원에서는 **RAG** 를 배웁니다. 오늘까지 익힌 **API·구조화 출력·정형화**와 지난 단원의 **임베딩**이 그 바탕이 됩니다. 수고하셨습니다!